In [1]:
import qnexus as qnx 

project = qnx.projects.get_or_create(name="CTCs")
qnx.context.set_active_project(project)

In [2]:
# Create a configuration to target the H1-1LE noiseless simulator
my_quantinuum_config = qnx.QuantinuumConfig(
    device_name="Helios-1E",
)

In [3]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from math import pi

def build_dctc(n: int, msg_params=None):
    """
    Step 1: Prepare message states on M[i] and swap them into C[i].
    (No entanglement, no scrambling/decoding, no measurements.)

    Args
    ----
    n : int
        Number of logical CTC qubits.
    msg_params : list[tuple[float, float, float]] | None
        Per-qubit single-qubit U gate parameters (theta, phi, lam) for M[i].
        If None, uses [(0,0,0)] for all i.

    Returns
    -------
    qc : QuantumCircuit
        Circuit containing only Step 1, with full 7n registers allocated
        in the order: C, E, R, G, M, A, Y, crA, crY, crC.
    """
    if msg_params is None:
      print("Please input the message parameters theta, phi, lambda")
    #     msg_params = [(0.0, 0.0, 0.0)] * n
    # assert len(msg_params) == n, "msg_params must have length n"

    # --- Allocate all registers now (to keep wire order consistent for later steps) ---
    C = QuantumRegister(n, 'C')   # CTC
    E = QuantumRegister(n, 'E')   # Early radiation
    R = QuantumRegister(n, 'R')   # Recent radiation
    G = QuantumRegister(n, 'G')   # Grover/projection ancilla
    M = QuantumRegister(n, 'M')   # Message
    A = QuantumRegister(n, 'A')   # Decoder ancilla
    Y = QuantumRegister(n, 'Y')   # Output
    crR = ClassicalRegister(n, 'crR')
    crG = ClassicalRegister(n, 'crG')
    crC = ClassicalRegister(n, 'crC')

    qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

    # --- Step 1a: Prepare messages on M[i] ---
    for i, (theta, phi, lam) in enumerate(msg_params):
        qc.u(theta, phi, lam, M[i])

    # --- Step 1b: Load messages into the CTC qubits ---
    for i in range(n):
        qc.swap(C[i], M[i])

    qc.barrier()  # delimiter: messages loaded into C

    # ---- EPR Pairs -----

    for i in range(n):
        # (E[i], M[i])
        qc.h(E[i])
        qc.cx(E[i], M[i])

        # (R[i], G[i])
        qc.h(R[i])
        qc.cx(R[i], G[i])

        # (A[i], Y[i])
        qc.h(A[i])
        qc.cx(A[i], Y[i])

    qc.barrier()

        # --- Scrambling Unitary (per qubit) ---
    for i in range(n):
        qc.cz(C[i], R[i])
        qc.cz(E[i], R[i])
        qc.cz(C[i], E[i])
        qc.h(C[i])
        qc.h(E[i])
        qc.h(R[i])
        qc.cz(C[i], R[i])
        qc.cz(C[i], E[i])
        qc.cz(E[i], R[i])
        qc.barrier()

    # --- Unitary Conjugate (Decoder) (per qubit) ---
    for i in range(n):
        qc.cz(A[i], G[i])
        qc.cz(M[i], A[i])
        qc.cz(G[i], M[i])
        qc.h(A[i])
        qc.h(M[i])
        qc.h(G[i])
        qc.cz(A[i], G[i])
        qc.cz(G[i], M[i])
        qc.cz(M[i], A[i])
        qc.barrier()

    # ---- Grover operator (R,G) ----

    for i in range(n):
      qc.rz(pi, R[i])
      qc.rx(pi, R[i])
      qc.rx(pi, G[i])
      qc.swap(R[i], G[i])
      qc.rz(pi, R[i])
      qc.barrier()


    # --- Unitary Transpose (Decoder) (per qubit) ---
    for i in range(n):
      qc.cz(A[i], G[i])
      qc.cz(M[i], A[i])
      qc.cz(G[i], M[i])
      qc.h(A[i])
      qc.h(M[i])
      qc.h(G[i])
      qc.cz(A[i], G[i])
      qc.cz(G[i], M[i])
      qc.cz(M[i], A[i])
      qc.barrier()

     # ---- Grover operator (A,Y) ----

    for i in range(n):
      qc.rz(pi, A[i])
      qc.rx(pi, A[i])
      qc.rx(pi, Y[i])
      qc.swap(A[i], Y[i])
      qc.rz(pi, A[i])
      qc.barrier()


   # --- Unitary Conjugate (Decoder) (per qubit) ---
    for i in range(n):
      qc.cz(A[i], G[i])
      qc.cz(M[i], A[i])
      qc.cz(G[i], M[i])
      qc.h(A[i])
      qc.h(M[i])
      qc.h(G[i])
      qc.cz(A[i], G[i])
      qc.cz(G[i], M[i])
      qc.cz(M[i], A[i])
      qc.barrier()



    # --- Bell Projection per qubit ---
    for i in range(n):
        qc.cx(R[i], G[i])
        qc.h(R[i])
        qc.measure(R[i], crR[i])
        qc.measure(G[i], crG[i])
        qc.barrier()

    # --- Getting back input message into C  ---


    for i in range(n):
        qc.swap(C[i], Y[i])

    for i, (theta, phi, lam) in enumerate(msg_params):
        qc.u(theta, phi, lam, C[i]).inverse()

    for i in range(n):
        qc.measure(C[i], crC[i])


    return qc

In [9]:
# from qiskit_aer import AerSimulator
# from qiskit import transpile
from pytket.extensions.qiskit.qiskit_convert import qiskit_to_tk

# 1. Build your circuit
msg_params = [(2.5349076035276403, 2.0022404587009195, 0.0),   # M[0]
              ]
qc = build_dctc(n=1, msg_params=msg_params)

# 2. Convert Qiskit → pytket
tk_circuit = qiskit_to_tk(qc)

# 3. Upload to Nexus
my_circuit_ref = qnx.circuits.upload(
    name="1QDCTC_circuit",
    circuit=tk_circuit,
    project=project,
)

In [25]:
"""
dctc_nexus_benchmark.py
=======================
Compiles build_dctc(n) for m = 1, 2, 3 on the Helios-1E emulator via
Quantinuum Nexus, extracts compiled circuit metrics, and produces the
table + scaling figure comparing logical vs compiled.

Usage
-----
    python dctc_nexus_benchmark.py

Requirements
------------
    pip install qnexus pytket pytket-qiskit matplotlib pandas
"""

import time
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import numpy as np

import qnexus as qnx
from quantinuum_schemas.models.backend_config import HeliosConfig
from pytket.extensions.qiskit import qiskit_to_tk

# ── YOUR PROJECT REF ──────────────────────────────────────────────────────────
project = qnx.projects.get(name="CTCs")   # <── change this
# Or if you already have a ref object in scope, just assign:
#   project = my_existing_project_ref

BACKEND_NAME = "Helios-1E"   # Helios emulator — uses HeliosConfig, not QuantinuumConfig

# ── Circuit builder (paste your build_dctc here or import it) ─────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from math import pi

def build_dctc(n: int, msg_params=None):
    if msg_params is None:
        msg_params = [(0.0, 0.0, 0.0)] * n
    assert len(msg_params) == n

    # Register names must start with a lowercase letter for QASM compatibility
    C = QuantumRegister(n, 'qc')
    E = QuantumRegister(n, 'qe')
    R = QuantumRegister(n, 'qr')
    G = QuantumRegister(n, 'qg')
    M = QuantumRegister(n, 'qm')
    A = QuantumRegister(n, 'qa')
    Y = QuantumRegister(n, 'qy')
    crR = ClassicalRegister(n, 'crr')
    crG = ClassicalRegister(n, 'crg')
    crC = ClassicalRegister(n, 'crc')

    qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

    for i, (theta, phi, lam) in enumerate(msg_params):
        qc.u(theta, phi, lam, M[i])
    for i in range(n):
        qc.swap(C[i], M[i])
    qc.barrier()

    for i in range(n):
        qc.h(E[i]); qc.cx(E[i], M[i])
        qc.h(R[i]); qc.cx(R[i], G[i])
        qc.h(A[i]); qc.cx(A[i], Y[i])
    qc.barrier()

    for i in range(n):
        qc.cz(C[i], R[i]); qc.cz(E[i], R[i]); qc.cz(C[i], E[i])
        qc.h(C[i]); qc.h(E[i]); qc.h(R[i])
        qc.cz(C[i], R[i]); qc.cz(C[i], E[i]); qc.cz(E[i], R[i])
        qc.barrier()

    for i in range(n):
        qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
        qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
        qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
        qc.barrier()

    for i in range(n):
        qc.rz(pi, R[i]); qc.rx(pi, R[i])
        qc.rx(pi, G[i]); qc.swap(R[i], G[i])
        qc.rz(pi, R[i]); qc.barrier()

    for i in range(n):
        qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
        qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
        qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
        qc.barrier()

    for i in range(n):
        qc.rz(pi, A[i]); qc.rx(pi, A[i])
        qc.rx(pi, Y[i]); qc.swap(A[i], Y[i])
        qc.rz(pi, A[i]); qc.barrier()

    for i in range(n):
        qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
        qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
        qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
        qc.barrier()

    for i in range(n):
        qc.cx(R[i], G[i]); qc.h(R[i])
        qc.measure(R[i], crR[i]); qc.measure(G[i], crG[i])
        qc.barrier()

    for i in range(n):
        qc.swap(C[i], Y[i])
    for i, (theta, phi, lam) in enumerate(msg_params):
        qc.u(theta, phi, lam, C[i]).inverse()
    for i in range(n):
        qc.measure(C[i], crC[i])

    return qc


# ── Logical metric extraction (from Qiskit DAG) ───────────────────────────────
def logical_metrics(qc) -> dict:
    from qiskit.converters import circuit_to_dag

    dag = circuit_to_dag(qc)
    d_logical = dag.depth()

    two_q_nodes = [n for n in dag.gate_nodes() if len(n.qargs) == 2]
    n2q = len(two_q_nodes)

    # Build 2q-only sub-circuit to measure entangling depth
    # node.op is correct in Qiskit 1.x (node.operation was removed)
    sub = qc.copy_empty_like()
    for node in dag.topological_op_nodes():
        if len(node.qargs) == 2:
            sub.append(node.op, node.qargs)
    d_2q = circuit_to_dag(sub).depth()

    return {"N_2q_logical": n2q, "d_logical": d_logical, "d_2q_logical": d_2q}


# ── Compiled metric extraction (from pytket circuit) ──────────────────────────
def compiled_metrics(tk_circuit) -> dict:
    from pytket.circuit import OpType

    # Helios native 2q gate is ZZPhase (and ZZMax which is a fixed ZZPhase)
    TWO_Q = {OpType.ZZMax, OpType.ZZPhase, OpType.PhasedISWAP, OpType.ISWAP,
             OpType.CX, OpType.CZ}   # include CX/CZ in case optimiser keeps them

    cmds = tk_circuit.get_commands()
    n2q_comp = sum(1 for c in cmds if c.op.type in TWO_Q)
    d_comp   = tk_circuit.depth()

    # 2q-only depth: clone circuit with only 2q gates
    from pytket import Circuit as TKCircuit
    sub = TKCircuit(tk_circuit.n_qubits, tk_circuit.n_bits)
    for c in cmds:
        if c.op.type in TWO_Q:
            sub.add_gate(c.op.type, c.op.params, c.args)
    d_2q_comp = sub.depth()

    return {"N_2q_compiled": n2q_comp, "d_compiled": d_comp, "d_2q_compiled": d_2q_comp}


# ── Main benchmark loop ───────────────────────────────────────────────────────
def run_benchmark(m_values=(1, 2, 3)):
    rows = []
    dummy = lambda m: [(0.5, 0.2, 0.1)] * m   # fixed non-trivial message state

    for m in m_values:
        print(f"\n{'='*55}")
        print(f"  m = {m}  ({7*m} qubits total)")
        print(f"{'='*55}")

        # ── 1. Build Qiskit circuit ──────────────────────────────────────────
        qc = build_dctc(n=m, msg_params=dummy(m))
        log = logical_metrics(qc)
        print(f"  Logical  →  N_2q={log['N_2q_logical']}  "
              f"d={log['d_logical']}  d_2q={log['d_2q_logical']}")

        # ── 2. Convert Qiskit → pytket ──────────────────────────────────────
        tk_circ = qiskit_to_tk(qc)

        # ── 3. Upload pytket circuit to Nexus ────────────────────────────────
        # qnx.circuits.upload requires a pytket Circuit object (not QASM string).
        # Register names are already lowercase (qc, qe, qr, ...) so pytket is happy.
        circuit_ref = qnx.circuits.upload(
            name=f"DCTC_m{m}",
            circuit=tk_circ,
            project=project,
        )
        print(f"  Uploaded circuit ref: {circuit_ref}")

        # ── 4. Compile on Nexus (no shots) ───────────────────────────────────
        # optimisation_level is a top-level arg to start_compile_job (not inside HeliosConfig).
        compile_job = qnx.start_compile_job(
            programs=circuit_ref,           # single ref, not a list
            backend_config=qnx.QuantinuumConfig(
                device_name=BACKEND_NAME,
                no_opt=False,               # ensure compiler runs
            ),
            name=f"DCTC_m{m}_compile",
            project=project,
        )
        print(f"  Compile job submitted: {compile_job}")

        # ── 5. Wait for compilation to finish ────────────────────────────────
        print(f"  Waiting for compilation...", end="", flush=True)
        while True:
            status = qnx.jobs.status(compile_job)
            if status.status in ("COMPLETED", "ERROR", "CANCELLED"):
                break
            print(".", end="", flush=True)
            time.sleep(5)
        print(f" {status.status}")

        if status.status != "COMPLETED":
            print(f"  [ERROR] Compilation failed for m={m}: {status}")
            rows.append({"m": m, **log,
                         "N_2q_compiled": None, "d_compiled": None, "d_2q_compiled": None})
            continue

        # ── 6. Retrieve compiled circuit ─────────────────────────────────────
        compiled_ref = qnx.jobs.results(compile_job)[0]
        compiled_tk  = qnx.circuits.download(compiled_ref)

        # ── 7. Extract compiled metrics ──────────────────────────────────────
        comp = compiled_metrics(compiled_tk)
        print(f"  Compiled →  N_2q={comp['N_2q_compiled']}  "
              f"d={comp['d_compiled']}  d_2q={comp['d_2q_compiled']}")

        rows.append({"m": m, **log, **comp})

    return pd.DataFrame(rows)


# ── Table figure ──────────────────────────────────────────────────────────────
def make_table(df, save_path="dctc_metrics_table.png"):
    col_keys = ["m",
                "N_2q_logical", "d_logical", "d_2q_logical",
                "N_2q_compiled", "d_compiled", "d_2q_compiled"]
    col_labels = [r"$m$",
                  r"$N_{2q}^{\rm logical}$", r"$d^{\rm logical}$", r"$d_{2q}^{\rm logical}$",
                  r"$N_{2q}^{\rm compiled}$", r"$d^{\rm compiled}$", r"$d_{2q}^{\rm compiled}$"]

    def fmt(v):
        return "–" if (v is None or (isinstance(v, float) and np.isnan(v))) else str(int(v))

    cell_data = [[fmt(row[k]) for k in col_keys] for _, row in df.iterrows()]

    fig, ax = plt.subplots(figsize=(13, 1.8 + 0.6 * len(df)))
    ax.axis("off")

    tbl = ax.table(cellText=cell_data, colLabels=col_labels,
                   cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(13)
    tbl.scale(1.0, 2.1)

    HDR, ROW_A, ROW_B, DIVDR = "#1f4e79", "#dce6f1", "#ffffff", "#2c7bb6"
    for j in range(len(col_labels)):
        tbl[0, j].set_facecolor(HDR)
        tbl[0, j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(df) + 1):
        bg = ROW_A if i % 2 == 0 else ROW_B
        for j in range(len(col_keys)):
            tbl[i, j].set_facecolor(bg)
    for i in range(len(df) + 1):
        tbl[i, 3].set_edgecolor(DIVDR)
        tbl[i, 4].set_edgecolor(DIVDR)

    ax.set_title(
        f"DCTC Circuit Resource Metrics  ·  Logical vs {BACKEND_NAME} Compiled",
        fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=200)
    print(f"\n[table]  saved → {save_path}")


# ── Scaling figure ────────────────────────────────────────────────────────────
def make_scaling_figure(df, save_path="dctc_metrics_scaling.png"):
    m_vals = df["m"].values
    metrics = [
        ("N_2q_logical", "N_2q_compiled", r"Two-qubit gate count $N_{2q}$"),
        ("d_logical",    "d_compiled",    r"Total circuit depth $d$"),
        ("d_2q_logical", "d_2q_compiled", r"Entangling depth $d_{2q}$"),
    ]
    CLR_L, CLR_C = "#2c7bb6", "#d7191c"

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle(
        r"DCTC Circuit Scaling — Logical vs " + BACKEND_NAME + r" Compiled  ($m=1,2,3$)",
        fontsize=13, fontweight="bold", y=1.02)

    for ax, (kl, kc, ylabel) in zip(axes, metrics):
        y_l = df[kl].values.astype(float)
        y_c = np.array([v if v is not None else np.nan
                        for v in df[kc].values], dtype=float)

        ax.plot(m_vals, y_l, "o-",  color=CLR_L, lw=2.5, ms=8, label="Logical")
        if not np.all(np.isnan(y_c)):
            ax.plot(m_vals, y_c, "s--", color=CLR_C, lw=2.5, ms=8,
                    label=f"Compiled ({BACKEND_NAME})")

        ax.set_xlabel(r"$m$  (CTC lanes)", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_xticks(m_vals)
        ax.yaxis.set_major_locator(ticker.MaxNLocator(integer=True))
        ax.legend(fontsize=9)
        ax.grid(True, linestyle="--", alpha=0.45)

    plt.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=200)
    print(f"[figure] saved → {save_path}")


# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    df = run_benchmark(m_values=[1, 2, 3])

    print("\n=== Final Metrics Table ===")
    print(df.to_string(index=False))

    make_table(df,          save_path="dctc_metrics_table.png")
    make_scaling_figure(df, save_path="dctc_metrics_scaling.png")
    print("\nAll done.")


  m = 1  (7 qubits total)
  Logical  →  N_2q=32  d=55  d_2q=21
  Uploaded circuit ref: id=UUID('39bed9f2-cba1-466d-b2a4-163431a98640') annotations=Annotations(name='DCTC_m1', description=None, properties=OrderedDict(), created=datetime.datetime(2026, 2, 25, 19, 38, 9, 213041, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 25, 19, 38, 9, 225175, tzinfo=TzInfo(0))) project=ProjectRef(id=UUID('ef977c2a-b90c-4a03-8283-96ddd2c72dd9'), annotations=Annotations(name='CTCs', description='D-CTCs and P-CTCs', properties=OrderedDict(), created=datetime.datetime(2026, 2, 9, 19, 29, 57, 91376, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 9, 20, 0, 19, 669887, tzinfo=TzInfo(0))), contents_modified=datetime.datetime(2026, 2, 25, 19, 36, 38, 986284, tzinfo=TzInfo(0)), archived=False, type='ProjectRef') type='CircuitRef'


ResourceCreateFailed: Failed to create resource with status code: 400, message: {"message":"Pytket circuits are not supported for this BackendConfig."}

In [15]:
import qnexus as qnx

# See all config classes available
print([x for x in dir(qnx) if 'Config' in x or 'config' in x.lower()])

# Also check what backends are actually listed
backends = qnx.devices.get_all().df()

print(backends)

['AerConfig', 'AerStateConfig', 'AerUnitaryConfig', 'BackendConfig', 'BraketConfig', 'IBMQConfig', 'IBMQEmulatorConfig', 'QuantinuumConfig', 'QulacsConfig', 'SeleneConfig', 'SelenePlusConfig', 'config', 'gpu_decoder_configs']
      backend_name                device_name  nexus_hosted  \
0              Aer              aer_simulator          True   
1         AerState  aer_simulator_statevector          True   
2       AerUnitary      aer_simulator_unitary          True   
3           Braket                        sv1          True   
4   Helios-1E-lite                       None          True   
5       Quantinuum                      H2-2E         False   
6       Quantinuum                      H2-1E         False   
7       Quantinuum                 Helios-1SC         False   
8       Quantinuum                     H2-1SC         False   
9       Quantinuum                  Helios-1E         False   
10      Quantinuum                       H2-1         False   
11      Quantinuum

In [17]:
print(qnx.SeleneConfig)

<class 'quantinuum_schemas.models.backend_config.SeleneConfig'>


In [21]:
help(qnx.QuantinuumConfig)

Help on class QuantinuumConfig in module quantinuum_schemas.models.backend_config:

class QuantinuumConfig(BaseBackendConfig)
 |  QuantinuumConfig(
 |      *,
 |      type: Literal['QuantinuumConfig'] = 'QuantinuumConfig',
 |      device_name: str,
 |      simulator: str = 'state-vector',
 |      machine_debug: bool = False,
 |      attempt_batching: bool = False,
 |      allow_implicit_swaps: bool = True,
 |      postprocess: bool = False,
 |      noisy_simulation: bool = True,
 |      target_2qb_gate: Optional[str] = None,
 |      user_group: Optional[str] = None,
 |      max_batch_cost: int = 2000,
 |      compiler_options: Optional[quantinuum_schemas.models.backend_config.QuantinuumCompilerOptions] = None,
 |      no_opt: bool = True,
 |      allow_2q_gate_rebase: bool = False,
 |      leakage_detection: bool = False,
 |      simplify_initial: bool = False,
 |      max_cost: Optional[int] = None,
 |      error_params: Optional[quantinuum_schemas.models.quantinuum_systems_noise.User

In [22]:
help(qnx.start_compile_job)

Help on function start_compile_job in module qnexus.client.jobs._compile:

start_compile_job(
    programs: Union[qnexus.models.references.CircuitRef, list[qnexus.models.references.CircuitRef]],
    backend_config: Annotated[Union[quantinuum_schemas.models.backend_config.AerConfig, quantinuum_schemas.models.backend_config.AerStateConfig, quantinuum_schemas.models.backend_config.AerUnitaryConfig, quantinuum_schemas.models.backend_config.BraketConfig, quantinuum_schemas.models.backend_config.QuantinuumConfig, quantinuum_schemas.models.backend_config.IBMQConfig, quantinuum_schemas.models.backend_config.IBMQEmulatorConfig, quantinuum_schemas.models.backend_config.QulacsConfig, quantinuum_schemas.models.backend_config.SeleneConfig, quantinuum_schemas.models.backend_config.SelenePlusConfig, quantinuum_schemas.models.backend_config.HeliosConfig], FieldInfo(annotation=NoneType, required=True, discriminator='type')],
    name: str,
    description: str = '',
    project: qnexus.models.reference

In [26]:
import qnexus as qnx
from quantinuum_schemas.models.backend_config import HeliosConfig

# Step 1: confirm your project
project = qnx.projects.get(name="CTCs")
print("Project:", project)

# Step 2: confirm the circuit uploaded earlier is findable
circuit_ref = qnx.circuits.get(name="DCTC_m1", project=project)
print("Circuit:", circuit_ref)

# Step 3: try the compile job with bare minimum args
compile_job = qnx.start_compile_job(
    programs=circuit_ref,
    backend_config=qnx.QuantinuumConfig(
        device_name="Helios-1E",
        no_opt=False,
    ),
    optimisation_level=2,
    name="DCTC_m1_compile_test",
    project=project,
)
print("Job submitted:", compile_job)

Project: id=UUID('ef977c2a-b90c-4a03-8283-96ddd2c72dd9') annotations=Annotations(name='CTCs', description='D-CTCs and P-CTCs', properties=OrderedDict(), created=datetime.datetime(2026, 2, 9, 19, 29, 57, 91376, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 9, 20, 0, 19, 669887, tzinfo=TzInfo(0))) contents_modified=datetime.datetime(2026, 2, 25, 19, 38, 9, 228182, tzinfo=TzInfo(0)) archived=False type='ProjectRef'


NoUniqueMatch: 

In [27]:
import qnexus as qnx

# Step 1: confirm login + project
project = qnx.projects.get(name="CTCs")
print("Project OK:", project.annotations.name)

# Step 2: see what a working compile job looks like for your account
# by checking any previously successful compile job (if any exist)
jobs = qnx.jobs.get_all(project=project)
for j in jobs:
    print(j)

Project OK: CTCs
id=UUID('cb5cacb6-1462-4a4b-9e6d-fe45dfde7ee2') annotations=Annotations(name='compile_async', description='', properties=OrderedDict(), created=datetime.datetime(2026, 2, 9, 21, 46, 15, 773602, tzinfo=TzInfo(0)), modified=datetime.datetime(2026, 2, 9, 21, 46, 54, 645073, tzinfo=TzInfo(0))) job_type=<JobType.COMPILE: 'compile'> last_status=<JobStatusEnum.COMPLETED: 'COMPLETED'> last_message='The job is completed.' last_status_detail=JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 2, 9, 21, 46, 54, 645073, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 2, 9, 21, 46, 22, 308636, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 2, 9, 21, 46, 15, 779334, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 2, 9, 21, 46, 50, 479877, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, 

In [29]:
import qnexus as qnx
from pytket.extensions.qiskit import qiskit_to_tk
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from math import pi

project = qnx.projects.get(name="CTCs")

# ── Build m=1 circuit ────────────────────────────────────────────────────────
n = 1
msg_params = [(0.5, 0.2, 0.1)]

C = QuantumRegister(n, 'qc')
E = QuantumRegister(n, 'qe')
R = QuantumRegister(n, 'qr')
G = QuantumRegister(n, 'qg')
M = QuantumRegister(n, 'qm')
A = QuantumRegister(n, 'qa')
Y = QuantumRegister(n, 'qy')
crR = ClassicalRegister(n, 'crr')
crG = ClassicalRegister(n, 'crg')
crC = ClassicalRegister(n, 'crc')

qc = QuantumCircuit(C, E, R, G, M, A, Y, crR, crG, crC)

for i, (theta, phi, lam) in enumerate(msg_params):
    qc.u(theta, phi, lam, M[i])
for i in range(n):
    qc.swap(C[i], M[i])
qc.barrier()
for i in range(n):
    qc.h(E[i]); qc.cx(E[i], M[i])
    qc.h(R[i]); qc.cx(R[i], G[i])
    qc.h(A[i]); qc.cx(A[i], Y[i])
qc.barrier()
for i in range(n):
    qc.cz(C[i], R[i]); qc.cz(E[i], R[i]); qc.cz(C[i], E[i])
    qc.h(C[i]); qc.h(E[i]); qc.h(R[i])
    qc.cz(C[i], R[i]); qc.cz(C[i], E[i]); qc.cz(E[i], R[i])
    qc.barrier()
for i in range(n):
    qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
    qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
    qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
    qc.barrier()
for i in range(n):
    qc.rz(pi, R[i]); qc.rx(pi, R[i])
    qc.rx(pi, G[i]); qc.swap(R[i], G[i])
    qc.rz(pi, R[i]); qc.barrier()
for i in range(n):
    qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
    qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
    qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
    qc.barrier()
for i in range(n):
    qc.rz(pi, A[i]); qc.rx(pi, A[i])
    qc.rx(pi, Y[i]); qc.swap(A[i], Y[i])
    qc.rz(pi, A[i]); qc.barrier()
for i in range(n):
    qc.cz(A[i], G[i]); qc.cz(M[i], A[i]); qc.cz(G[i], M[i])
    qc.h(A[i]); qc.h(M[i]); qc.h(G[i])
    qc.cz(A[i], G[i]); qc.cz(G[i], M[i]); qc.cz(M[i], A[i])
    qc.barrier()
for i in range(n):
    qc.cx(R[i], G[i]); qc.h(R[i])
    qc.measure(R[i], crR[i]); qc.measure(G[i], crG[i])
    qc.barrier()
for i in range(n):
    qc.swap(C[i], Y[i])
for i, (theta, phi, lam) in enumerate(msg_params):
    qc.u(theta, phi, lam, C[i]).inverse()
for i in range(n):
    qc.measure(C[i], crC[i])

# ── Convert and upload ───────────────────────────────────────────────────────
tk_circ = qiskit_to_tk(qc)
print("pytket circuit qubits:", tk_circ.n_qubits, "bits:", tk_circ.n_bits)

circuit_ref = qnx.circuits.upload(
    name="DCTC_m1_v2",
    circuit=tk_circ,
    project=project,
)
print("Uploaded:", circuit_ref.annotations.name)

# ── Compile against H1-1LE first (known to work for your account) ────────────
compile_job = qnx.start_compile_job(
    programs=circuit_ref,
    backend_config=qnx.QuantinuumConfig(device_name="H1-1LE"),
    name="DCTC_m1_v2_compile",
    project=project,
)
print("Compile job submitted:", compile_job.annotations.name, compile_job.id)

pytket circuit qubits: 7 bits: 3
Uploaded: DCTC_m1_v2
Compile job submitted: DCTC_m1_v2_compile 21b13c1c-e0f9-4834-93c4-f61247b45255


In [32]:
import qnexus as qnx

project = qnx.projects.get(name="CTCs")

compile_job_helios = qnx.jobs.get(name="DCTC_m1_v2_compile", project=project)
print("Found job:", compile_job_helios.annotations.name)
print("Status:", compile_job_helios.last_status)

Found job: DCTC_m1_v2_compile
Status: JobStatusEnum.COMPLETED


In [33]:
import time

# ── Wait for job to complete ─────────────────────────────────────────────────
print("Waiting for compilation...", end="", flush=True)
while True:
    status = qnx.jobs.status(compile_job_helios)
    if status.status.value in ("COMPLETED", "ERROR", "CANCELLED"):
        break
    print(".", end="", flush=True)
    time.sleep(5)
print(f" {status.status.value}")

# ── Retrieve compiled circuit ────────────────────────────────────────────────
compiled_ref = qnx.jobs.results(compile_job_helios)[0]
compiled_tk = qnx.circuits.download(compiled_ref)
print("Downloaded compiled circuit")
print("  Qubits:", compiled_tk.n_qubits)
print("  Bits:  ", compiled_tk.n_bits)

# ── Extract compiled metrics ─────────────────────────────────────────────────
from pytket.circuit import OpType

# Helios native 2q gates
TWO_Q = {OpType.ZZMax, OpType.ZZPhase, OpType.PhasedISWAP,
         OpType.ISWAP, OpType.CX, OpType.CZ}

cmds = compiled_tk.get_commands()

# N_2q_compiled
n2q_compiled = sum(1 for c in cmds if c.op.type in TWO_Q)

# d_compiled (total depth)
d_compiled = compiled_tk.depth()

# d_2q_compiled (entangling depth only)
from pytket import Circuit as TKCircuit
sub = TKCircuit(compiled_tk.n_qubits, compiled_tk.n_bits)
for c in cmds:
    if c.op.type in TWO_Q:
        sub.add_gate(c.op.type, c.op.params, c.args)
d_2q_compiled = sub.depth()

print(f"\n  N_2q_compiled  = {n2q_compiled}")
print(f"  d_compiled     = {d_compiled}")
print(f"  d_2q_compiled  = {d_2q_compiled}")

Waiting for compilation... COMPLETED


AttributeError: module 'qnexus.client.circuits' has no attribute 'download'

In [34]:
print([x for x in dir(qnx.circuits) if not x.startswith('_')])

['Annotations', 'Any', 'BackendConfig', 'Circuit', 'CircuitRef', 'CreateAnnotations', 'CreatorFilter', 'DataframableList', 'ExecutionProgram', 'NameFilter', 'NexusIterator', 'PaginationFilter', 'Params', 'ProjectRef', 'ProjectRefFilter', 'PropertiesDict', 'PropertiesFilter', 'QuantinuumConfig', 'ScopeFilter', 'ScopeFilterEnum', 'SortFilter', 'SortFilterEnum', 'TimeFilter', 'UUID', 'Union', 'cast', 'circuit_dict_from_pytket1_dict', 'cost', 'datetime', 'get', 'get_active_project', 'get_all', 'get_nexus_client', 'handle_fetch_errors', 'merge_project_from_context', 'merge_properties_from_context', 'merge_scope_from_context', 'qnx_exc', 'update', 'upload', 'warn']


In [35]:
compiled_ref = qnx.jobs.results(compile_job_helios)[0]
print(type(compiled_ref))
print(dir(compiled_ref))

<class 'qnexus.models.references.CompilationResultRef'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__

In [36]:
compiled_tk = compiled_ref.get_output()
print(type(compiled_tk))
print("Qubits:", compiled_tk.n_qubits)
print("Bits:  ", compiled_tk.n_bits)

<class 'qnexus.models.references.CircuitRef'>


AttributeError: 'CircuitRef' object has no attribute 'n_qubits'

In [37]:
print(type(compiled_tk))
print([x for x in dir(qnx.circuits) if not x.startswith('_')])

# Try getting the pytket circuit via qnx.circuits.get
circuit = qnx.circuits.get(id=compiled_tk.id)
print(type(circuit))
print(dir(circuit))

<class 'qnexus.models.references.CircuitRef'>
['Annotations', 'Any', 'BackendConfig', 'Circuit', 'CircuitRef', 'CreateAnnotations', 'CreatorFilter', 'DataframableList', 'ExecutionProgram', 'NameFilter', 'NexusIterator', 'PaginationFilter', 'Params', 'ProjectRef', 'ProjectRefFilter', 'PropertiesDict', 'PropertiesFilter', 'QuantinuumConfig', 'ScopeFilter', 'ScopeFilterEnum', 'SortFilter', 'SortFilterEnum', 'TimeFilter', 'UUID', 'Union', 'cast', 'circuit_dict_from_pytket1_dict', 'cost', 'datetime', 'get', 'get_active_project', 'get_all', 'get_nexus_client', 'handle_fetch_errors', 'merge_project_from_context', 'merge_properties_from_context', 'merge_scope_from_context', 'qnx_exc', 'update', 'upload', 'warn']
<class 'qnexus.models.references.CircuitRef'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__form

In [38]:
compiled_tk = compiled_ref.get_output().download_circuit()
print(type(compiled_tk))
print("Qubits:", compiled_tk.n_qubits)
print("Bits:  ", compiled_tk.n_bits)

<class 'pytket._tket.circuit.Circuit'>
Qubits: 7
Bits:   3


In [39]:
from pytket.circuit import OpType

# Helios native 2q gates
TWO_Q = {OpType.ZZMax, OpType.ZZPhase, OpType.PhasedISWAP, OpType.ISWAP, OpType.CX, OpType.CZ}

cmds = compiled_tk.get_commands()

# N_2q_compiled
n2q_compiled = sum(1 for c in cmds if c.op.type in TWO_Q)

# d_compiled
d_compiled = compiled_tk.depth()

# d_2q_compiled
from pytket import Circuit as TKCircuit
sub = TKCircuit(compiled_tk.n_qubits, compiled_tk.n_bits)
for c in cmds:
    if c.op.type in TWO_Q:
        sub.add_gate(c.op.type, c.op.params, c.args)
d_2q_compiled = sub.depth()

print(f"m=1 compiled metrics:")
print(f"  N_2q_compiled = {n2q_compiled}")
print(f"  d_compiled    = {d_compiled}")
print(f"  d_2q_compiled = {d_2q_compiled}")

m=1 compiled metrics:
  N_2q_compiled = 28
  d_compiled    = 66
  d_2q_compiled = 18


In [40]:
from pytket.circuit import OpType
from collections import Counter

# Full gate inventory
gate_counts = Counter(str(c.op.type) for c in cmds)
print("Gate inventory:")
for gate, count in sorted(gate_counts.items()):
    print(f"  {gate}: {count}")

# 1q gate count and depth
ONE_Q = {OpType.Rz, OpType.Rx, OpType.Ry, OpType.H, OpType.X, OpType.Y, 
         OpType.Z, OpType.S, OpType.T, OpType.U1, OpType.U2, OpType.U3,
         OpType.PhasedX, OpType.ZZMax, OpType.Measure}
n1q_compiled = sum(1 for c in cmds if len(c.args) == 1 and c.op.type != OpType.Measure)

# Measurement count
n_measurements = sum(1 for c in cmds if c.op.type == OpType.Measure)

# Total gate count (excluding barriers)
n_gates_total = sum(1 for c in cmds if c.op.type != OpType.Barrier)

print(f"\nm=1 full compiled metrics:")
print(f"  Total gates (ex. barriers) = {n_gates_total}")
print(f"  1q gates                   = {n1q_compiled}")
print(f"  2q gates                   = {n2q_compiled}")
print(f"  Measurements               = {n_measurements}")
print(f"  Total depth                = {d_compiled}")
print(f"  2q depth                   = {d_2q_compiled}")
print(f"  Qubits                     = {compiled_tk.n_qubits}")
print(f"  Bits                       = {compiled_tk.n_bits}")

Gate inventory:
  OpType.Barrier: 9
  OpType.Measure: 3
  OpType.PhasedX: 66
  OpType.Rz: 16
  OpType.ZZPhase: 28

m=1 full compiled metrics:
  Total gates (ex. barriers) = 113
  1q gates                   = 82
  2q gates                   = 28
  Measurements               = 3
  Total depth                = 66
  2q depth                   = 18
  Qubits                     = 7
  Bits                       = 3
